In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def main():
    data = load_breast_cancer()
    x_train, x_test, y_train, y_test = train_test_split(
        data.data,
        data.target,
        test_size=0.2,
        random_state=42,
        stratify=data.target,
    )


    scaler = StandardScaler()
    x_train_scaled = scaler.fit_transform(x_train)
    x_test_scaled = scaler.transform(x_test)

    original_model = RandomForestClassifier(n_estimators=200, random_state=42)
    original_model.fit(x_train_scaled, y_train)
    original_accuracy = accuracy_score(y_test, original_model.predict(x_test_scaled))

    pca = PCA(n_components=0.95, random_state=42)
    x_train_pca = pca.fit_transform(x_train_scaled)
    x_test_pca = pca.transform(x_test_scaled)

    pca_model = RandomForestClassifier(n_estimators=200, random_state=42)
    pca_model.fit(x_train_pca, y_train)
    pca_accuracy = accuracy_score(y_test, pca_model.predict(x_test_pca))

    print(f"Accuracy на 30 признаках: {original_accuracy:.4f}")
    print(f"Accuracy после PCA:       {pca_accuracy:.4f}")
    print(f"Сохранено компонент: {pca.n_components_} из {x_train.shape[1]}")
    print(f"Изменение accuracy: {pca_accuracy - original_accuracy:+.4f}")

    try:
        import matplotlib.pyplot as plt
        import umap
    except ImportError:
        print("\nДля UMAP установите зависимости: pip install umap-learn matplotlib")
        return


    x_scaled = StandardScaler().fit_transform(data.data)
    embedding = umap.UMAP(n_components=2, random_state=42).fit_transform(x_scaled)
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=data.target,
        cmap="coolwarm",
        s=18,
        alpha=0.8,
    )
    plt.title("Breast Cancer: визуализация UMAP")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.legend(handles=scatter.legend_elements()[0], labels=["malignant", "benign"])
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    main()


- С 30 признаками accuracy равна `0.9561`; после PCA, сохранившего 95% дисперсии в 10 компонентах, — `0.9386`. Снижение составило `0.0175`, то есть 1.75 процентного пункта.
- Стандартизация занимает `O(nd)`. PCA через SVD требует примерно `O(nd·min(n,d))` времени и `O(nd)` памяти. Обучение Random Forest растёт примерно пропорционально числу деревьев; в этой задаче модель обучается на 30 признаках или на 10 компонентах.